# Target Coverage for Logarithmic Windows

This notebook is a companion to [`logging_scheduler.ipynb`](logging_scheduler.ipynb). It zooms in on
one parameter of `logarithmic_windows()`: **`target_coverage`**.

**Background.** In a batch-size sweep at a *fixed token budget*, `max_steps` shrinks linearly as
batch size grows (`max_steps = total_blocks // batch_size`). A fixed `base_window` can't compensate
for that: window *count* only grows with `log10(max_steps)`, so realized analysis coverage
(`len(schedule.steps) / max_steps`) balloons as batch size increases. `target_coverage` fixes this by
deriving `base_window` from `max_steps` instead, so coverage stays roughly constant across the sweep.

**What this notebook does:**
1. Reproduces the exact before/after coverage table from the 2026-08-21 thesis-log entry, against the
   real packed-corpus token count, as a sanity check that we're using `logarithmic_windows()` the same way.
2. Extends that anchor sweep to a denser range of batch sizes.
3. Answers the question behind the fix: **how many training tokens does one measurement window cost,
   and how does that change with batch size** -- with `target_coverage` on vs. off.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from perspic.logger import logarithmic_windows

## Fixed token budget

We hold the *training data* fixed -- same packed corpus, same sequence length -- and only vary batch
size. `max_steps` is then a consequence of the budget, not something chosen independently:

```
max_steps(batch_size) = total_blocks // batch_size
```

The numbers below are the real packed-corpus size from the thesis-log entry (`meta.json`:
608,617,592 train tokens at `max_seq_len=512`).

In [ ]:
TOTAL_TRAIN_TOKENS = 608_617_592  # packed-corpus size, from meta.json
MAX_SEQ_LEN = 512
TOTAL_BLOCKS = TOTAL_TRAIN_TOKENS // MAX_SEQ_LEN  # one step of batch_size=1 consumes one block

REFERENCE_BATCH_SIZE = 128  # the sweep's existing wall-time reference batch size
BASE_WINDOW = 50  # fixed base_window used before the fix (target_coverage "off")
TARGET_COVERAGE = 0.2154  # calibrated so target_coverage reproduces base_window=50 at bs=128 exactly

print(f"total_blocks = {TOTAL_BLOCKS:,}")

## Reproducing the thesis-log anchor table

Before extending anything, check that `logarithmic_windows()` reproduces the exact coverage numbers
from the log entry for its five batch sizes. "`target_coverage` off" means passing a fixed
`base_window`; "`target_coverage` on" means passing `target_coverage` instead and letting it derive
`base_window` from `max_steps`.

In [ ]:
def sweep_coverage(batch_sizes, base_window=BASE_WINDOW, target_coverage=TARGET_COVERAGE,
                    points_per_decade=10):
    """Build fixed-vs-target-coverage schedules across a batch-size sweep at a fixed token budget."""
    rows = []
    for bs in batch_sizes:
        max_steps = TOTAL_BLOCKS // bs
        fixed = logarithmic_windows(
            max_steps=max_steps, points_per_decade=points_per_decade, base_window=base_window
        )
        target = logarithmic_windows(
            max_steps=max_steps, points_per_decade=points_per_decade, target_coverage=target_coverage
        )
        # adaptive_scale=0 (the default) means every window in a schedule has the same width
        fixed_width = len(next(iter(fixed.windows.values())))
        target_width = len(next(iter(target.windows.values())))
        tokens_per_step = bs * MAX_SEQ_LEN

        rows.append({
            "batch_size": bs,
            "max_steps": max_steps,
            "window_width_fixed": fixed_width,
            "window_width_target": target_width,
            "coverage_fixed_%": 100 * len(fixed.steps) / max_steps,
            "coverage_target_%": 100 * len(target.steps) / max_steps,
            "tokens_per_step": tokens_per_step,
            "tokens_per_window_fixed": fixed_width * tokens_per_step,
            "tokens_per_window_target": target_width * tokens_per_step,
            "analysis_tokens_total_fixed": len(fixed.steps) * tokens_per_step,
            "analysis_tokens_total_target": len(target.steps) * tokens_per_step,
        })
    return pd.DataFrame(rows)


anchor_batch_sizes = [4, 32, 128, 256, 1024]
anchor_df = sweep_coverage(anchor_batch_sizes)
display(anchor_df[["batch_size", "max_steps", "coverage_fixed_%", "coverage_target_%"]].round(2))

These match the 2026-08-21 log entry exactly: coverage collapses from **0.59% -> 49.40%** across
the sweep with a fixed `base_window=50` down to **8.55% -> 13.28%** with `target_coverage=0.2154` --
and bs=128 is bit-for-bit identical between the two, since `target_coverage` was calibrated to
reproduce that exact anchor schedule.

## A denser batch-size sweep

The log entry's table only has five points. Filling in the powers of two between them (which include
all five anchor points above) makes the trend, and the crossover at bs=128, easier to see.

In [ ]:
batch_sizes = [4, 8, 16, 32, 64, 128, 256, 512, 1024]
df = sweep_coverage(batch_sizes)
display(df.round(2))

## Coverage: the spread that motivated the fix

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(df["batch_size"], df["coverage_fixed_%"], "o-", label="target_coverage off (base_window=50)")
ax.plot(df["batch_size"], df["coverage_target_%"], "o-", label=f"target_coverage on ({TARGET_COVERAGE})")
ax.axvline(REFERENCE_BATCH_SIZE, color="gray", linestyle=":", linewidth=1,
           label=f"anchor bs={REFERENCE_BATCH_SIZE}")
ax.set_xscale("log", base=2)
ax.set_xticks(batch_sizes)
ax.set_xticklabels(batch_sizes)
ax.set_xlabel("batch size")
ax.set_ylabel("realized coverage (%)")
ax.set_title("Analysis-window coverage vs. batch size, fixed token budget")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## The actual question: tokens per measurement window

Coverage is a useful summary, but it hides the mechanism. Each scheduled step processes
`batch_size * max_seq_len` tokens, and -- since `adaptive_scale=0` here, so every window in a
schedule has the same width -- a whole measurement window costs:

```
tokens_per_window = window_width * batch_size * max_seq_len
```

With a **fixed `base_window`**, `window_width` doesn't know about `batch_size` at all, so
`tokens_per_window` scales linearly with batch size purely because each step got heavier. With
**`target_coverage`**, `window_width` is derived from `max_steps` (which shrinks as batch size
grows), and that shrinkage largely cancels the growth in tokens-per-step.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
ax.plot(df["batch_size"], df["window_width_fixed"], "o-", label="target_coverage off")
ax.plot(df["batch_size"], df["window_width_target"], "o-", label="target_coverage on")
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xticks(batch_sizes)
ax.set_xticklabels(batch_sizes)
ax.set_xlabel("batch size")
ax.set_ylabel("window width (steps)")
ax.set_title("Steps per measurement window")
ax.grid(True, alpha=0.3, which="both")
ax.legend()

ax = axes[1]
ax.plot(df["batch_size"], df["tokens_per_window_fixed"], "o-", label="target_coverage off")
ax.plot(df["batch_size"], df["tokens_per_window_target"], "o-", label="target_coverage on")
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xticks(batch_sizes)
ax.set_xticklabels(batch_sizes)
ax.set_xlabel("batch size")
ax.set_ylabel("tokens per measurement window")
ax.set_title("Training tokens consumed per window")
ax.grid(True, alpha=0.3, which="both")
ax.legend()

plt.tight_layout()
plt.show()

Across the bs=4..1024 sweep (a 256x range in batch size):

- **`target_coverage` off**: window width is pinned at 50 steps, so tokens-per-window scales with
  batch size directly -- **~102K -> ~26.2M tokens**, a 256x spread.
- **`target_coverage` on**: window width falls from 1164 steps (bs=4) to 8 steps (bs=1024), almost
  exactly canceling the 256x growth in tokens-per-step -- so tokens-per-window only moves
  **~2.4M -> ~4.2M tokens**, under 2x.

So `target_coverage` isn't just holding *coverage* constant -- it's holding the *token cost of a
single measurement* roughly constant, by trading window width for batch size as the budget shifts
between them.

## Total analysis token spend across a full run

Zooming back out: multiplying coverage by the total token budget gives the total number of training
tokens analyzed over the *whole* run, in tokens rather than as a percentage -- a more concrete number
if you're budgeting analysis compute.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(df["batch_size"], df["analysis_tokens_total_fixed"], "o-", label="target_coverage off")
ax.plot(df["batch_size"], df["analysis_tokens_total_target"], "o-", label="target_coverage on")
ax.axhline(TOTAL_TRAIN_TOKENS, color="gray", linestyle=":", linewidth=1,
           label="full corpus (608.6M tokens)")
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xticks(batch_sizes)
ax.set_xticklabels(batch_sizes)
ax.set_xlabel("batch size")
ax.set_ylabel("total tokens spent on analysis")
ax.set_title("Total analysis-token spend across the whole run")
ax.grid(True, alpha=0.3, which="both")
ax.legend()
plt.tight_layout()
plt.show()

## Bonus: combining with `adaptive_scale`

`target_coverage` only sets the *base* window width for a schedule; `adaptive_scale` still grows
individual windows later in training on top of that base
(`window_size = base_window + adaptive_scale * log10(center)`). The two combine independently:
`target_coverage` picks the starting width for a given batch size, `adaptive_scale` then grows
windows as `center` increases within that schedule.

In [ ]:
demo_bs = 128
demo_max_steps = TOTAL_BLOCKS // demo_bs

for scale in (0.0, 2.0):
    schedule = logarithmic_windows(
        max_steps=demo_max_steps, target_coverage=TARGET_COVERAGE, adaptive_scale=scale
    )
    widths = [len(steps) for steps in schedule.windows.values()]
    print(f"adaptive_scale={scale}: window width ranges {min(widths)}-{max(widths)} steps "
          f"over {len(schedule.windows)} windows")

## Takeaways

- At a fixed token budget, `max_steps` shrinks linearly with batch size, but a fixed `base_window`
  doesn't shrink with it -- so both coverage *and* the token cost of one measurement window balloon
  as batch size grows.
- `target_coverage` derives `base_window` from `max_steps`, which keeps both roughly constant across
  a batch-size sweep instead.
- `target_coverage` is calibrated once, at whatever batch size you already trust (here, bs=128, the
  sweep's wall-time reference point) -- pick it so the anchor schedule matches what you had before,
  then let it drive the rest of the sweep.

See [`perspic/logger.py`](../perspic/logger.py)'s `logarithmic_windows()` docstring and
[`tests/unit/test_logger.py`](../tests/unit/test_logger.py)'s `TestTargetCoverage` for the
parameter's exact semantics and edge cases (e.g. the minimum-window-width clamp at very low
`target_coverage`, or `max_steps <= 0`).